# AI Lab: Global (LHS-PRCC) vs Local Sensitivity Analysis

**Book companion exercise**

**Considerations exercised:** 11 (correct sensitivity analysis)
**Estimated runtime:** ~30 minutes
**Companion audit:** [`10_local_vs_global_sensitivity.md`](../ai-audit/10_local_vs_global_sensitivity.md)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/exercises/ai-lab/10_global_sensitivity_lab.ipynb)


## What this lab does

Compares local sensitivity indices (computed via partial derivatives at a nominal point) with global PRCC indices (computed from Latin Hypercube sampling across full uncertainty ranges). Demonstrates that for the SIR final-attack-rate, the two approaches give substantially different rankings when parameter uncertainty is large.

## Setup

In [ ]:
import sys, os
# AI Lab notebooks live in exercises/ai-lab/ and need access to shared/
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'python')))
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy.optimize import minimize
import math
from shared import book_style, BOOK_COLORS
from shared.parameters import baseline_chapter_07
book_style()
rng = np.random.default_rng(42)


## Step 1 — Define the model and the local sensitivities

Final attack rate (AR) for the SIR model satisfies $1 - AR = \exp(-\mathcal{R}_0 \cdot AR)$, with $\mathcal{R}_0 = c_I \beta / \gamma$. Local sensitivities are computed at a nominal point $(c_I = 14, \beta = 0.04, \gamma = 0.2)$.

In [ ]:
from scipy.optimize import brentq

def AR_from_R0(R0):
    if R0 <= 1.0:
        return 0.0
    # f(0+) > 0; f(1-) is also > 0 for large R0 because exp(-R0) is tiny.
    # The root is near 1 - exp(-R0) for large R0; bracket carefully.
    # Use a tight upper-bracket choice: any value where f changes sign
    # is to the right of the actual root.
    a_lo = 1e-9
    a_hi = 1 - 1e-12
    f_lo = 1 - a_lo - math.exp(-R0*a_lo)
    f_hi = 1 - a_hi - math.exp(-R0*a_hi)
    if f_lo * f_hi > 0:
        # Root is very close to 1; binary-search a tighter upper bracket
        return 1.0 - 1e-12  # sentinel: AR effectively 1
    return brentq(lambda a: 1 - a - math.exp(-R0*a), a_lo, a_hi)

# Nominal parameters
nom = dict(c_I=14.0, beta=0.04, gamma=0.2)
R0_nom = nom['c_I'] * nom['beta'] / nom['gamma']
AR_nom = AR_from_R0(R0_nom)
print(f'Nominal R_0 = {R0_nom:.3f}, AR = {AR_nom:.3f}')

# Local sensitivity via finite differences
h = 1e-4
local_sens = {}
for p in ('c_I', 'beta', 'gamma'):
    perturbed = dict(nom)
    perturbed[p] = nom[p] * (1 + h)
    R0_p = perturbed['c_I'] * perturbed['beta'] / perturbed['gamma']
    AR_p = AR_from_R0(R0_p)
    # Normalized log-sensitivity
    local_sens[p] = (math.log(AR_p) - math.log(AR_nom)) / (math.log(perturbed[p]) - math.log(nom[p]))

print('\nLocal sensitivity indices:')
for p, s in local_sens.items():
    print(f'  S^AR_{p} = {s:+.3f}')


## Step 2 — Latin Hypercube sample across realistic uncertainty

For an early-pandemic setting, parameter uncertainty is typically order-of-magnitude. We sample $c_I \in [5, 30]$, $\beta \in [0.01, 0.10]$, $\gamma \in [0.05, 0.40]$:

In [ ]:
from scipy.stats import qmc, rankdata

n_samples = 1000
sampler = qmc.LatinHypercube(d=3, seed=42)
raw = sampler.random(n=n_samples)
lo = np.array([5.0, 0.01, 0.05])
hi = np.array([30.0, 0.10, 0.40])
samples = lo + raw * (hi - lo)

AR_out = np.zeros(n_samples)
for i in range(n_samples):
    cI, b, g = samples[i]
    R0 = cI * b / g
    AR_out[i] = AR_from_R0(R0)


## Step 3 — Compute PRCC indices

Partial Rank Correlation Coefficient (PRCC) measures the rank correlation between each parameter and the output, after removing linear effects of the other parameters. This gives a *global* sensitivity ranking.

In [ ]:
def prcc(samples, output):
    n, k = samples.shape
    # Rank-transform
    R_in = np.column_stack([rankdata(samples[:, j]) for j in range(k)])
    R_out = rankdata(output)
    prcc_vals = np.zeros(k)
    for j in range(k):
        # Residualize R_in[:, j] and R_out against the other ranks
        others = np.delete(np.arange(k), j)
        X = np.column_stack([R_in[:, others], np.ones(n)])
        # Residual of R_in[:, j]
        beta_j = np.linalg.lstsq(X, R_in[:, j], rcond=None)[0]
        resid_j = R_in[:, j] - X @ beta_j
        # Residual of R_out
        beta_y = np.linalg.lstsq(X, R_out, rcond=None)[0]
        resid_y = R_out - X @ beta_y
        # Pearson correlation of residuals
        prcc_vals[j] = np.corrcoef(resid_j, resid_y)[0, 1]
    return prcc_vals

prcc_vals = prcc(samples, AR_out)
param_names = ['c_I', 'beta', 'gamma']

print('Global PRCC indices (full LHS sample):')
for p, v in zip(param_names, prcc_vals):
    print(f'  PRCC_{p} = {v:+.3f}')


## Step 4 — Compare local vs global rankings

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.5))
x = np.arange(3)
w = 0.35
local_vals = [local_sens[p] for p in param_names]
ax.bar(x - w/2, local_vals, w, color=BOOK_COLORS['secondary'],
       edgecolor='black', label='Local sensitivity (at nominal)')
ax.bar(x + w/2, prcc_vals, w, color=BOOK_COLORS['primary'],
       edgecolor='black', label='Global PRCC (LHS sample)')
ax.axhline(0, color='black', lw=0.6)
ax.set_xticks(x); ax.set_xticklabels(param_names)
ax.set_ylabel('Sensitivity index')
ax.set_title('Local vs global sensitivity for SIR final attack rate')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

# Verification: if local and global differ in absolute ranking,
# this is the operational consequence of large uncertainty
local_rank = np.argsort([abs(v) for v in local_vals])[::-1]
global_rank = np.argsort([abs(v) for v in prcc_vals])[::-1]
print(f'Local-method ranking:  {[param_names[i] for i in local_rank]}')
print(f'Global-method ranking: {[param_names[i] for i in global_rank]}')


## What's next

- For this model, the local and global rankings agree (the AR-vs-R_0 function is approximately log-linear over the sampled range)
- For nonlinear models with sharper threshold behavior, the rankings can differ substantially — see book Chapter 7 §7.5 for examples
- Companion AI Audit: `../ai-audit/10_local_vs_global_sensitivity.md`